# TimeGAN (Forex OHLCV) — Improved Training Notebook

**What this notebook fixes vs typical TimeGAN runs**
- Discriminator strengthened (`n_critic`), TTUR (separate LR for G/D)
- Gradient penalty to prevent D collapse
- Volatility + tail losses to reduce "too-smooth" synthetic series
- Early stopping by **quality metrics**, not by GAN losses
- Hard OHLC sanity checks and post-processing

You only need to change the **CONFIG** cell paths and a few hyperparameters.


In [ ]:
# =========================
# 0) CONFIG
# =========================
DATA_PATH = r"/content/cleaned_no_weekend.csv"   # <-- change to your file (CSV). Columns: time, open, high, low, close, volume
TIME_COL = "time"

O_COL, H_COL, L_COL, C_COL, V_COL = "open", "high", "low", "close", "volume"

SEQ_LEN = 48             # window length (e.g., 48 for 2 days if hourly)
BATCH_SIZE = 256
NOISE_DIM = 16
HIDDEN_DIM = 64
NUM_LAYERS = 2

# Training
PRETRAIN_EMBEDDER_EPOCHS = 100
SUPERVISOR_EPOCHS = 100
JOINT_EPOCHS = 600

# GAN stabilization
N_CRITIC = 3             # D steps per 1 G step
LR_G = 2e-4
LR_D = 6e-4              # TTUR: D faster than G
BETA1, BETA2 = 0.5, 0.9

# Regularization / penalties
GP_LAMBDA = 10.0         # gradient penalty weight
VOL_LAMBDA = 5.0         # volatility/abs-return matching weight
TAIL_LAMBDA = 3.0        # tail quantile matching weight

# Early stopping
EARLYSTOP_PATIENCE = 40  # epochs
SAVE_DIR = "/mnt/data/timegan_improved_ckpt"

# Repro
SEED = 42


In [ ]:
# =========================
# 1) Imports + Setup
# =========================
import os, math, json, random
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers

import matplotlib.pyplot as plt

os.makedirs(SAVE_DIR, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

seed_everything(SEED)

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# =========================

In [ ]:
# =========================
# 2) Load Data (Fixing missing df)
# =========================
if 'df' not in locals():
    print(f"Loading data from {DATA_PATH}...")
    df = pd.read_csv(DATA_PATH)
    # Sort by time if column exists
    if TIME_COL in df.columns:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL])
        df = df.sort_values(TIME_COL).reset_index(drop=True)
    print(f"Data loaded. Shape: {df.shape}")

# =========================
# 3) Feature Engineering (helps realism)
# =========================
eps = 1e-12

# log return
df["logret"] = np.log(df[C_COL]).diff().fillna(0.0)

# range metrics
df["hl_range"] = (df[H_COL] - df[L_COL]) / (df[C_COL] + eps)

# ATR(14) approximation
tr1 = (df[H_COL] - df[L_COL]).abs()
tr2 = (df[H_COL] - df[C_COL].shift(1)).abs()
tr3 = (df[L_COL] - df[C_COL].shift(1)).abs()
tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1).fillna(0.0)
df["atr14"] = tr.rolling(14).mean().fillna(method="bfill").fillna(0.0) / (df[C_COL] + eps)

# Realized vol (rolling std of logret)
df["realized_vol"] = df["logret"].rolling(24).std().fillna(method="bfill").fillna(0.0)

# RSI(14)
delta = df[C_COL].diff().fillna(0.0)
gain = (delta.where(delta > 0, 0.0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
rs = (gain / (loss + eps)).fillna(0.0)
df["rsi14"] = (100 - (100 / (1 + rs))).fillna(50.0) / 100.0  # scale 0..1

# Volume ratio
if V_COL in df.columns:
    df["volume_ratio"] = (df[V_COL] / (df[V_COL].rolling(24).mean() + eps)).fillna(1.0)
else:
    df[V_COL] = 0.0
    df["volume_ratio"] = 1.0

# Build final feature set for TimeGAN
FEATURES = [O_COL, H_COL, L_COL, C_COL, V_COL, "logret", "hl_range", "atr14", "realized_vol", "rsi14", "volume_ratio"]
X = df[FEATURES].astype("float32").values
X.shape

/tmp/ipython-input-531431626.py:29: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["atr14"] = tr.rolling(14).mean().fillna(method="bfill").fillna(0.0) / (df[C_COL] + eps)
/tmp/ipython-input-531431626.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["realized_vol"] = df["logret"].rolling(24).std().fillna(method="bfill").fillna(0.0)


(42634, 11)

In [ ]:
# =========================
# 4) Scaling (robust, per-feature)
# =========================
# Robust scaling: (x - median) / (IQR)
med = np.median(X, axis=0)
q1 = np.percentile(X, 25, axis=0)
q3 = np.percentile(X, 75, axis=0)
iqr = (q3 - q1)
iqr[iqr < 1e-9] = 1.0

Xn = (X - med) / iqr

# Clip extreme outliers (stabilizes GAN)
Xn = np.clip(Xn, -10, 10).astype("float32")

# Save scaler params
scaler = {"median": med.tolist(), "iqr": iqr.tolist(), "features": FEATURES}
with open(os.path.join(SAVE_DIR, "scaler.json"), "w") as f:
    json.dump(scaler, f)

Xn.shape


(42634, 11)

In [ ]:
# =========================
# 5) Windowing -> Dataset
# =========================
def make_windows(arr, seq_len):
    n = len(arr)
    out = []
    for i in range(0, n - seq_len + 1):
        out.append(arr[i:i+seq_len])
    return np.stack(out, axis=0)

W = make_windows(Xn, SEQ_LEN)
print("Windows:", W.shape)  # (N, T, D)

dataset = tf.data.Dataset.from_tensor_slices(W)
dataset = dataset.shuffle(min(len(W), 20000), seed=SEED, reshuffle_each_iteration=True)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

DATA_DIM = W.shape[-1]


Windows: (42587, 48, 11)


## 6) TimeGAN Models (TF2/Keras)
This is a compact TimeGAN implementation:
- Embedder/Recovery: maps real data <-> latent space
- Generator/Supervisor: generates latent trajectories
- Discriminator: distinguishes real latent vs synthetic latent

We add:
- **Gradient penalty** on discriminator
- **Volatility loss**: match mean(|logret|) and mean(logret^2)
- **Tail loss**: match 1% / 99% quantiles of logret (batchwise)


In [ ]:
# =========================
# 6) Model Builders
# =========================
def rnn_stack(hidden_dim, num_layers, return_sequences=True, name_prefix="rnn"):
    seq = tf.keras.Sequential(name=name_prefix)
    for i in range(num_layers):
        seq.add(layers.GRU(hidden_dim, return_sequences=True, name=f"{name_prefix}_gru{i}"))
    return seq

class TimeGAN(tf.keras.Model):
    def __init__(self, data_dim, hidden_dim, num_layers, noise_dim):
        super().__init__()
        self.data_dim = data_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.noise_dim = noise_dim

        # Embedder and Recovery
        self.embedder = tf.keras.Sequential([
            rnn_stack(hidden_dim, num_layers, name_prefix="emb"),
            layers.TimeDistributed(layers.Dense(hidden_dim, activation="sigmoid"))
        ], name="embedder")

        self.recovery = tf.keras.Sequential([
            rnn_stack(hidden_dim, num_layers, name_prefix="rec"),
            layers.TimeDistributed(layers.Dense(data_dim, activation=None))
        ], name="recovery")

        # Generator and Supervisor (latent)
        self.generator = tf.keras.Sequential([
            rnn_stack(hidden_dim, num_layers, name_prefix="gen"),
            layers.TimeDistributed(layers.Dense(hidden_dim, activation="sigmoid"))
        ], name="generator")

        self.supervisor = tf.keras.Sequential([
            rnn_stack(hidden_dim, max(1, num_layers-1), name_prefix="sup"),
            layers.TimeDistributed(layers.Dense(hidden_dim, activation="sigmoid"))
        ], name="supervisor")

        # Discriminator on latent
        self.discriminator = tf.keras.Sequential([
            rnn_stack(hidden_dim, num_layers, name_prefix="dis"),
            layers.TimeDistributed(layers.Dense(1, activation=None))
        ], name="discriminator")

    def embed(self, x):
        return self.embedder(x)

    def recover(self, h):
        return self.recovery(h)

    def gen_latent(self, z):
        e_hat = self.generator(z)
        h_hat = self.supervisor(e_hat)
        return e_hat, h_hat

    def sup_latent(self, h):
        return self.supervisor(h)

    def disc(self, h):
        return self.discriminator(h)

timegan = TimeGAN(DATA_DIM, HIDDEN_DIM, NUM_LAYERS, NOISE_DIM)


In [ ]:
# =========================
# 7) Losses + Optimizers
# =========================
mse = tf.keras.losses.MeanSquaredError()
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

opt_e = tf.keras.optimizers.Adam(LR_G, beta_1=BETA1, beta_2=BETA2)
opt_g = tf.keras.optimizers.Adam(LR_G, beta_1=BETA1, beta_2=BETA2)
opt_d = tf.keras.optimizers.Adam(LR_D, beta_1=BETA1, beta_2=BETA2)

def discriminator_gp(disc_fn, real_h, fake_h):
    # WGAN-GP style gradient penalty on interpolates
    alpha = tf.random.uniform([tf.shape(real_h)[0], 1, 1], 0.0, 1.0)
    interp = alpha * real_h + (1.0 - alpha) * fake_h
    with tf.GradientTape() as tape:
        tape.watch(interp)
        out = disc_fn(interp)  # (B,T,1)
        # reduce to scalar per-sample
        out_mean = tf.reduce_mean(out, axis=[1,2])
    grads = tape.gradient(out_mean, interp)
    grads = tf.reshape(grads, [tf.shape(grads)[0], -1])
    gp = tf.reduce_mean((tf.norm(grads, axis=1) - 1.0) ** 2)
    return gp

def batch_quantile(x, q):
    # x: (B,T) -> quantile over all elements
    flat = tf.reshape(x, [-1])
    k = tf.cast(tf.round(q * tf.cast(tf.size(flat)-1, tf.float32)), tf.int32)
    values = tf.sort(flat)
    return values[tf.clip_by_value(k, 0, tf.size(values)-1)]

def vol_tail_losses(x_real, x_fake, feat_index_logret):
    # use logret feature from scaled space? better: use unscaled approx -> but we only have normalized in training.
    # We'll match in normalized logret domain; still helps tails and vol.
    r = x_real[:,:,feat_index_logret]
    f = x_fake[:,:,feat_index_logret]

    # volatility proxy losses
    vol1 = tf.abs(tf.reduce_mean(tf.abs(r)) - tf.reduce_mean(tf.abs(f)))
    vol2 = tf.abs(tf.reduce_mean(tf.square(r)) - tf.reduce_mean(tf.square(f)))
    vol_loss = vol1 + vol2

    # tail quantiles (1% and 99%)
    q01_r = batch_quantile(r, 0.01)
    q99_r = batch_quantile(r, 0.99)
    q01_f = batch_quantile(f, 0.01)
    q99_f = batch_quantile(f, 0.99)
    tail_loss = tf.abs(q01_r - q01_f) + tf.abs(q99_r - q99_f)

    return vol_loss, tail_loss

LOGRET_IDX = FEATURES.index("logret")


In [ ]:
# =========================
# 8) Training Steps
# =========================
@tf.function
def train_embedder_step(x):
    # Reconstruction loss: x -> h -> x_tilde
    with tf.GradientTape() as tape:
        h = timegan.embed(x)
        x_tilde = timegan.recover(h)
        e_loss = mse(x, x_tilde)
    vars_e = timegan.embedder.trainable_variables + timegan.recovery.trainable_variables
    grads = tape.gradient(e_loss, vars_e)
    opt_e.apply_gradients(zip(grads, vars_e))
    return e_loss

@tf.function
def train_supervisor_step(x):
    with tf.GradientTape() as tape:
        h = timegan.embed(x)
        h_sup = timegan.sup_latent(h)
        s_loss = mse(h[:,1:,:], h_sup[:,:-1,:])
    vars_s = timegan.supervisor.trainable_variables
    grads = tape.gradient(s_loss, vars_s)
    opt_g.apply_gradients(zip(grads, vars_s))
    return s_loss

@tf.function
def train_discriminator_step(x, z):
    # Discriminator on latent trajectories
    with tf.GradientTape() as tape:
        h = timegan.embed(x)
        e_hat, h_hat = timegan.gen_latent(z)

        # logits
        y_real = timegan.disc(h)
        y_fake = timegan.disc(h_hat)

        # BCE loss (stabilizes in practice for TimeGAN)
        d_loss = bce(tf.ones_like(y_real), y_real) + bce(tf.zeros_like(y_fake), y_fake)

        # GP
        gp = discriminator_gp(timegan.disc, h, h_hat)
        d_loss_total = d_loss + GP_LAMBDA * gp

    vars_d = timegan.discriminator.trainable_variables
    grads = tape.gradient(d_loss_total, vars_d)
    opt_d.apply_gradients(zip(grads, vars_d))
    return d_loss, gp, d_loss_total

@tf.function
def train_generator_step(x, z):
    with tf.GradientTape() as tape:
        h = timegan.embed(x)
        e_hat, h_hat = timegan.gen_latent(z)

        # adversarial loss (want D predict real on fake)
        y_fake = timegan.disc(h_hat)
        g_adv = bce(tf.ones_like(y_fake), y_fake)

        # supervised loss in latent space (time consistency)
        h_sup = timegan.sup_latent(h)
        g_sup = mse(h[:,1:,:], h_sup[:,:-1,:])

        # recovery to data space for additional realism
        x_hat = timegan.recover(h_hat)

        # volatility + tail losses on feature logret
        vol_loss, tail_loss = vol_tail_losses(x, x_hat, LOGRET_IDX)

        g_total = g_adv + 100.0 * tf.sqrt(g_sup + 1e-8) + VOL_LAMBDA * vol_loss + TAIL_LAMBDA * tail_loss

    vars_g = (timegan.generator.trainable_variables +
              timegan.supervisor.trainable_variables +
              timegan.embedder.trainable_variables +
              timegan.recovery.trainable_variables)

    grads = tape.gradient(g_total, vars_g)
    opt_g.apply_gradients(zip(grads, vars_g))
    return g_total, g_adv, g_sup, vol_loss, tail_loss


In [ ]:
# =========================
# 9) Quality Metrics (for early stopping)
# =========================
def synth_batch(x, n_batches=1):
    # generate synthetic windows in normalized feature space
    out = []
    for _ in range(n_batches):
        z = tf.random.uniform((BATCH_SIZE, SEQ_LEN, NOISE_DIM), 0, 1)
        _, h_hat = timegan.gen_latent(z)
        x_hat = timegan.recover(h_hat)
        out.append(x_hat.numpy())
    return np.concatenate(out, axis=0)

def metric_suite(real_windows, fake_windows, logret_idx):
    r = real_windows[:,:,logret_idx].reshape(-1)
    f = fake_windows[:,:,logret_idx].reshape(-1)

    # match std and kurtosis-ish via tail/quantiles
    def q(a, p):
        return np.quantile(a, p)

    m = {}
    m["mean_abs"] = float(abs(np.mean(np.abs(r)) - np.mean(np.abs(f))))
    m["mse_std"] = float((np.std(r) - np.std(f))**2)
    m["q01"] = float(abs(q(r,0.01) - q(f,0.01)))
    m["q99"] = float(abs(q(r,0.99) - q(f,0.99)))

    # composite score (lower is better)
    m["score"] = m["mean_abs"] + 0.5*m["mse_std"] + 2.0*(m["q01"] + m["q99"])
    return m


In [ ]:
# =========================
# 10) Phase 1 — Embedder Pretrain
# =========================
hist = {"e_loss": [], "s_loss": [], "g_total": [], "d_loss": [], "gp": [], "metrics": []}

for epoch in range(PRETRAIN_EMBEDDER_EPOCHS):
    losses = []
    for x in dataset:
        losses.append(float(train_embedder_step(x).numpy()))
    hist["e_loss"].append(np.mean(losses))
    if (epoch+1) % 20 == 0:
        print(f"[Embedder] {epoch+1}/{PRETRAIN_EMBEDDER_EPOCHS}  e_loss={hist['e_loss'][-1]:.6f}")

print("Done.")


[Embedder] 20/100  e_loss=0.002302
[Embedder] 40/100  e_loss=0.001184
[Embedder] 60/100  e_loss=0.000816
[Embedder] 80/100  e_loss=0.000627
[Embedder] 100/100  e_loss=0.000516
Done.


In [ ]:
# =========================
# 11) Phase 2 — Supervisor Pretrain
# =========================
for epoch in range(SUPERVISOR_EPOCHS):
    losses = []
    for x in dataset:
        losses.append(float(train_supervisor_step(x).numpy()))
    hist["s_loss"].append(np.mean(losses))
    if (epoch+1) % 20 == 0:
        print(f"[Supervisor] {epoch+1}/{SUPERVISOR_EPOCHS}  s_loss={hist['s_loss'][-1]:.6f}")

print("Done.")


In [ ]:
# =========================
# 12) Phase 3 — Joint Training (Improved)
# =========================
ckpt = tf.train.Checkpoint(model=timegan, opt_e=opt_e, opt_g=opt_g, opt_d=opt_d)
manager = tf.train.CheckpointManager(ckpt, SAVE_DIR, max_to_keep=3)

best_score = 1e18
best_epoch = -1
bad_count = 0

# Keep a fixed sample of real windows for stable evaluation
real_eval = W[np.random.RandomState(SEED).choice(len(W), size=min(4096, len(W)), replace=False)]

for epoch in range(JOINT_EPOCHS):
    g_totals, g_advs, g_sups, vls, tls = [], [], [], [], []
    d_losses, gps = [], []

    for x in dataset:
        # D steps
        for _ in range(N_CRITIC):
            z = tf.random.uniform((BATCH_SIZE, SEQ_LEN, NOISE_DIM), 0, 1)
            # Temporarily removing @tf.function from train_discriminator_step for debugging
            # This will allow eager execution and expose the actual error if any.
            d_loss, gp, d_total = train_discriminator_step(x, z)
            d_losses.append(float(d_loss.numpy()))
            gps.append(float(gp.numpy()))

        # G step
        z = tf.random.uniform((BATCH_SIZE, SEQ_LEN, NOISE_DIM), 0, 1)
        g_total, g_adv, g_sup, vol_loss, tail_loss = train_generator_step(x, z)

        g_totals.append(float(g_total.numpy()))
        g_advs.append(float(g_adv.numpy()))
        g_sups.append(float(g_sup.numpy()))
        vls.append(float(vol_loss.numpy()))
        tls.append(float(tail_loss.numpy()))

    # Evaluate every 10 epochs
    if (epoch+1) % 10 == 0:
        fake_eval = synth_batch(None, n_batches=max(1, 4096 // BATCH_SIZE))
        m = metric_suite(real_eval[:len(fake_eval)], fake_eval, LOGRET_IDX)
        hist["metrics"].append((epoch+1, m))

        print(f"[Joint] {epoch+1}/{JOINT_EPOCHS} "
              f"G={np.mean(g_totals):.3f} (adv={np.mean(g_advs):.3f}, sup={np.mean(g_sups):.4f}, vol={np.mean(vls):.4f}, tail={np.mean(tls):.4f}) | "
              f"D={np.mean(d_losses):.3f} gp={np.mean(gps):.3f} | score={m['score']:.6f}")

        # Early stopping on quality score
        if m["score"] < best_score:
            best_score = m["score"]
            best_epoch = epoch+1
            bad_count = 0
            manager.save()
            with open(os.path.join(SAVE_DIR, "best_metrics.json"), "w") as f:
                json.dump({"epoch": best_epoch, "best_score": best_score, "metrics": m}, f, indent=2)
        else:
            bad_count += 10
            if bad_count >= EARLYSTOP_PATIENCE:
                print(f"Early stop at epoch {epoch+1}. Best epoch: {best_epoch}  best_score={best_score:.6f}")
                break

print("Training finished. Best epoch:", best_epoch, "best_score:", best_score)


TypeError: cannot unpack non-iterable NoneType object

In [ ]:
# =========================
# 13) Load Best Checkpoint
# =========================

# Ensure ckpt and manager are defined if the kernel was restarted or skipped previous cells
if 'manager' not in locals() or 'ckpt' not in locals():
    print("Initializing CheckpointManager...")
    ckpt = tf.train.Checkpoint(model=timegan, opt_e=opt_e, opt_g=opt_g, opt_d=opt_d)
    manager = tf.train.CheckpointManager(ckpt, SAVE_DIR, max_to_keep=3)

latest = manager.latest_checkpoint
print("Latest ckpt:", latest)
if latest:
    ckpt.restore(latest).expect_partial()
    print(f"Restored from {latest}")
else:
    print("No checkpoint found. Starting fresh or continue training...")

In [ ]:
# =========================
# 14) Generate Synthetic Data (windows -> continuous series)
# =========================
def unscale(xn, med, iqr):
    return xn * iqr + med

def generate_windows(n_windows=2000):
    outs = []
    steps = math.ceil(n_windows / BATCH_SIZE)
    for _ in range(steps):
        z = tf.random.uniform((BATCH_SIZE, SEQ_LEN, NOISE_DIM), 0, 1)
        _, h_hat = timegan.gen_latent(z)
        x_hat = timegan.recover(h_hat).numpy()
        outs.append(x_hat)
    xw = np.concatenate(outs, axis=0)[:n_windows]
    return xw

syn_w = generate_windows(4000)

# Unscale
syn_w_un = unscale(syn_w, med, iqr)

# Keep only OHLCV columns for candle reconstruction
idxO, idxH, idxL, idxC, idxV = [FEATURES.index(c) for c in [O_COL,H_COL,L_COL,C_COL,V_COL]]
syn_ohlcv = syn_w_un[:,:, [idxO,idxH,idxL,idxC,idxV]]

syn_ohlcv.shape


In [ ]:
# =========================
# 15) Stitch windows (simple overlap-avg) + OHLC sanity post-process
# =========================
def stitch_windows(windows, overlap=True):
    # windows: (N,T,D)
    N,T,D = windows.shape
    if not overlap:
        return windows.reshape(-1, D)

    # overlap-add with averaging
    out_len = N + T - 1
    out = np.zeros((out_len, D), dtype=np.float64)
    cnt = np.zeros((out_len, 1), dtype=np.float64)

    for i in range(N):
        out[i:i+T] += windows[i]
        cnt[i:i+T] += 1.0

    out /= np.maximum(cnt, 1.0)
    return out.astype(np.float32)

syn_series = stitch_windows(syn_ohlcv, overlap=True)  # (L,5)
syn_df = pd.DataFrame(syn_series, columns=[O_COL,H_COL,L_COL,C_COL,V_COL])

# OHLC constraints
syn_df[H_COL] = np.maximum(syn_df[H_COL].values, np.maximum(syn_df[O_COL].values, syn_df[C_COL].values))
syn_df[L_COL] = np.minimum(syn_df[L_COL].values, np.minimum(syn_df[O_COL].values, syn_df[C_COL].values))
syn_df = syn_df[syn_df[H_COL] >= syn_df[L_COL]].reset_index(drop=True)

syn_df.head(), syn_df.shape


In [ ]:
# =========================
# 16) Quick Diagnostics (real vs synthetic)
# =========================
real_ohlcv = df[[O_COL,H_COL,L_COL,C_COL,V_COL]].iloc[:len(syn_df)].reset_index(drop=True)

def logret(x):
    return np.log(x[C_COL]).diff().fillna(0.0).values

r_lr = logret(real_ohlcv)
s_lr = logret(syn_df)

plt.figure()
plt.plot(r_lr[:500], label="Real")
plt.plot(s_lr[:500], label="Synthetic")
plt.axhline(0, ls="--", alpha=0.5)
plt.title("Log Returns (first 500)")
plt.legend()
plt.show()

# Hist
plt.figure()
plt.hist(r_lr, bins=120, alpha=0.5, label="Real", density=True)
plt.hist(s_lr, bins=120, alpha=0.5, label="Synthetic", density=True)
plt.title("Logret Distribution")
plt.legend()
plt.show()

# Tail check
for q in [0.001, 0.01, 0.99, 0.999]:
    print("q", q, "real", np.quantile(r_lr,q), "syn", np.quantile(s_lr,q))


In [ ]:
# =========================
# 17) Save Synthetic CSV
# =========================
out_csv = os.path.join(SAVE_DIR, "synthetic_ohlcv.csv")
syn_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


## Notes / knobs you will tune
- If synthetic is still too smooth: increase `VOL_LAMBDA` and `TAIL_LAMBDA`, increase `N_CRITIC`, reduce `LR_G`.
- If training becomes unstable: reduce `LR_D`, increase `GP_LAMBDA`, lower `BATCH_SIZE`.
- Best quality is usually near early-stop epoch, not at max epoch.
